# GT3 / GT300 source training (research only)

Run from the RaceEngineer repository with Python, `duckdb`, `pandas`, `numpy`, `pypdf`, and `xgboost`. This notebook executes the source-specific scripts; raw downloads and results stay in the Git-ignored `training/pit_strategy/.local_train/`.

- IMSA GTD/GTDPRO: one-lap-ahead pace regression. `est_tire_age` is excluded because it is a post-race heuristic.
- SUPER GT Suzuka 2024/2025: official GT300 GT3 lap timing; external pace check, no physical tyre wear label.
- Assetto Corsa GT3: 60-second simulator tyre-condition regression; the four source files have unusable lap counters.

The ingest pipeline tags these sources as `gt` using homologation, official entry lists, or dataset-card class; F1 research stays in `f1`. GT3/GT300 source class remains available for later profile-specific evaluation. Unknown car IDs stay `unknown` until source metadata confirms their class.

None of these sources supplies counterfactual optimal pit-lap labels. All model manifests remain `deployment_ready=false`; this notebook does not approve or install a RaceEngineer pit Ranker.


In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "training/pit_strategy/train_imsa_pace.py").is_file())
SCRIPTS = ROOT / "training/pit_strategy"
for args in (("train_imsa_pace.py",), ("train_supergt_gt300.py", "--year", "2024"),
             ("train_supergt_gt300.py", "--year", "2025"), ("train_ac_wear.py",)):
    subprocess.run([sys.executable, str(SCRIPTS / args[0]), *args[1:]], cwd=ROOT, check=True)


## Research metrics

Read the saved manifests and run an external-domain check. The SUPER GT PDFs lack caution and traffic flags, so their `pace_candidate` filter is heuristic. Keep its result separate from the held-out IMSA test.


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb

LOCAL = SCRIPTS / ".local_train"
imsa = json.loads((LOCAL / "imsa_gt3/manifest.json").read_text(encoding="utf-8"))
wear = json.loads((LOCAL / "ac_wear/metrics.json").read_text(encoding="utf-8"))
print("IMSA 2026 test:", imsa["metrics"]["test"])
print("AC 60-second condition LOSO:", {k: wear[k] for k in
      ("persistence_mae_pp", "xgboost_mae_pp", "improvement_over_persistence_pct")})

model = xgb.XGBRegressor()
model.load_model(LOCAL / "imsa_gt3/imsa_gt3_pace_delta_xgb.json")
transfer = {}
for year in (2024, 2025):
    source = LOCAL / f"supergt_gt300/supergt_suzuka{year}_gt300_gt3_laps.csv"
    laps = pd.read_csv(source, dtype={"car": str}).sort_values(
        ["event_id", "car", "stint", "lap"]).reset_index(drop=True)
    laps["stint_lap"] = laps.groupby(["event_id", "car", "stint"]).cumcount() + 1
    grouped = laps.groupby(["event_id", "car", "stint", "driver"], sort=False)
    for offset in (1, 2):
        laps[f"previous_{offset}_lap"] = grouped["lap"].shift(offset)
        laps[f"previous_{offset}_time"] = grouped["lap_time_s"].shift(offset)
        laps[f"previous_{offset}_clean"] = grouped["pace_candidate"].shift(offset).fillna(False).astype(bool)
    prior1 = laps["previous_1_clean"] & laps["previous_1_lap"].eq(laps["lap"] - 1)
    prior2 = prior1 & laps["previous_2_clean"] & laps["previous_2_lap"].eq(laps["lap"] - 2)
    laps["rolling_3_lap_s"] = (laps["lap_time_s"] + laps["previous_1_time"].where(prior1, 0)
                               + laps["previous_2_time"].where(prior2, 0)) / (1 + prior1.astype(int) + prior2.astype(int))
    laps["next_lap"] = grouped["lap"].shift(-1)
    laps["next_lap_time_s"] = grouped["lap_time_s"].shift(-1)
    laps["next_clean"] = grouped["pace_candidate"].shift(-1).fillna(False).astype(bool)
    pairs = laps[laps["pace_candidate"] & laps["next_clean"]
                 & laps["next_lap"].eq(laps["lap"] + 1)].copy()
    pairs["current_lap_time_s"] = pairs["lap_time_s"]
    actual = pairs["next_lap_time_s"].to_numpy()
    estimates = {
        "xgboost": pairs["current_lap_time_s"].to_numpy()
                    + model.predict(pairs[imsa["features"]]),
        "previous_lap": pairs["current_lap_time_s"].to_numpy(),
        "rolling_three": pairs["rolling_3_lap_s"].to_numpy(),
    }
    transfer[str(year)] = {
        "pairs": len(pairs),
        "errors_s": {name: {"mae": float(np.mean(np.abs(actual - estimate))),
                             "median_ae": float(np.median(np.abs(actual - estimate)))}
                     for name, estimate in estimates.items()},
    }
print("IMSA to SUPER GT transfer:", json.dumps(transfer, indent=2))
(LOCAL / "supergt_gt300/transfer_metrics.json").write_text(
    json.dumps({"scope": "external-domain descriptive check; caution/traffic labels unavailable",
                "model_sha256": imsa["model_sha256"], "results": transfer}, indent=2), encoding="utf-8")
